In [1]:
import os 
import getpass
from dotenv import load_dotenv
import pydantic
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
#
#llm.call('Hello')
#
load_dotenv()
#
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
import chromadb
from chromadb.config import Settings
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

class CustomKnowledgeStorage(KnowledgeStorage):

    def __init__(self, persist_directory: str, embedder=None, collection_name=None):
        self.persist_directory = persist_directory
        self.embedder = embedder
        self.collection_name = collection_name
        super().__init__(embedder=embedder, collection_name=collection_name)
    
    def initialize_knowledge_storage(self):
        chroma_client = chromadb.PersistentClient(
            path=self.persist_directory,
            settings=Settings(allow_reset=True),
        )
        self.app = chroma_client
        try:
            collection_name = (
                "knowledge" if not self.collection_name else self.collection_name
            )
            self.collection = self.app.get_or_create_collection(
                name=collection_name,
                embedding_function=self.embedder
            )
        except Exception as e:
            raise Exception(f"Failed to create or get collection: {e}")

In [ ]:
from agents import CodeInterpreterTool
from crewai_tools import CodeDocsSearchTool, DOCXSearchTool, GithubSearchTool, PDFSearchTool, ScrapeWebsiteTool, WebsiteSearchTool




In [ ]:
from crewai.tools import tool
from crewai_tools import PDFSearchTool

pydantic.SkipValidation

# 1. Initialize the tool
from crewai_tools import PDFSearchTool

# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
@tool
def pdf_search_tool(query : str):
        """_summary
            Searches the pdf documents and returns results.
        """
        pdf_tool = PDFSearchTool(
            pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
            config={
                "embedding_model": {
                    "provider": "openai",
                    "config": {
                        "model": "nomic-embed-text",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "embedder": {
                    "provider": "openai",
                    "config": {
                        "model": "all-minilm",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        "persist_directory":"agentic-ai/chromadb",
                        "allow_reset":"true",
                        "is_persistent":"true",
                    }
                },
            }
        )

        return pdf_tool.run(query)


In [3]:
results = pdf_search_tool.run("Cloudera")
results

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


"Relevant Content:\n\nimproved release frequency from monthly to bi-weekly, ensuring on-time delivery against stringent regulatory \n\ndeadlines. \n\n \n\n• Led Data Engineering Chapter for IB Markets Regulatory Reporting, governing 5+ regulatory \n\nframeworks (MIFID-II, RTS23, EUBR/UKBR, SSD, SFTR) — overseeing ingestion, reconciliation, and distribution \n\nof deal-store data for ~10 million+ daily trade & transaction records with full audit trail and zero regulatory \n\nbreach incidents across a 3-year delivery window. \n\n \n\n\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 

In [9]:

# 2. Define the Agent
researcher = Agent(
    role='Resume Analyst',
    goal='Extract key skills from the provided resume',
    backstory='Expert at analyzing candidate skills and identifying experience.',
    tools=[pdf_search_tool],
    verbose=True,
    llm=llm
)

# 3. Define the Task
task = Task(
    description='What is the core skills for candidate ?',
    expected_output='A summary of candidate with his core skills.',
    agent=researcher
)

# 4. Run the Crew
crew = Crew(
    agents=[researcher], 
    tasks=[task],
    verbose=True,
    planning=True,
    planning_llm=llm
)
result = crew.kickoff()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f593d4fd-e677-44c0-bbb6-7ef9a12ada53                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-04-16 11:39:28][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - What is the core skills for candidate ?                                        │
│                  "task_description": What is the core skills for candidate ?                                    │
│                  "task_expected_output": A summary of candidate with his core skills.                           │
│                  "agent": Resume Analyst                                                                        │
│                  "agent_goal": Extract key skills from the provided resume                                      │
│                  "task_tools": [Tool(name='pdf_search_tool', description='Tool Name: pdf_search_tool\nTool      │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Pdf_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: _summary\n            Searches the pdf documents and       │
│  returns results.\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Pdf_Search_Tool'>,        │
│  description_updated=False, cache_function=<function _default_cache_function at 0x7a68fce5f6a0>,                │
│  result_as_answer=False, max_usage_count=None, current_usage_count=0, func=<function pdf_search_tool at         │
│  0x7a68d414a200>, tool_type='crewai.tools.base_tool.Tool')]                                                     │
│                  "agent_tools": [name='pdf_search_tool' description='Tool Name: pdf_search_tool\nTool           │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Pdf_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: _summary\n            Searches the pdf documents and       │
│  returns results.\n        ' env_vars=[] args_schema=<class 'crewai.tools.base_tool.Pdf_Search_Tool'>           │
│  description_updated=False cache_function=<function _default_cache_function at 0x7a68fce5f6a0>                  │
│  result_as_answer=False max_usage_count=None current_usage_count=0 func=<function pdf_search_tool at            │
│  0x7a68d414a200> tool_type='crewai.tools.base_tool.Tool']                                                       │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  ID: 9885005f-97df-4d09-a73c-3616f7fa9fdd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - What is the core skills for candidate ?                                        │
│                  "task_description": What is the core skills for candidate ?                                    │
│                  "task_expected_output": A summary of candidate with his core skills.                           │
│                  "agent": Resume Analyst                                                                        │
│                  "agent_goal": Extract key skills from the provided resume                                      │
│                  "task_tools": [Tool(name='pdf_search_tool', description='Tool Name: pdf_search_tool\nTool      │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Pdf_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: _summary\n            Searches the pdf documents and       │
│  returns results.\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Pdf_Search_Tool'>,        │
│  description_updated=False, cache_function=<function _default_cache_function at 0x7a68fce5f6a0>,                │
│  result_as_answer=False, max_usage_count=None, current_usage_count=0, func=<function pdf_search_tool at         │
│  0x7a68d414a200>, tool_type='crewai.tools.base_tool.Tool')]                                                     │
│                  "agent_tools": [name='pdf_search_tool' description='Tool Name: pdf_search_tool\nTool           │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Pdf_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: _summary\n            Searches the pdf documents and       │
│  returns results.\n        ' env_vars=[] args_schema=<class 'crewai.tools.base_tool.Pdf_Search_Tool'>           │
│  description_updated=False cache_function=<function _default_cache_function at 0x7a68fce5f6a0>                  │
│  result_as_answer=False max_usage_count=None current_usage_count=0 func=<function pdf_search_tool at            │
│  0x7a68d414a200> tool_type='crewai.tools.base_tool.Tool']                                                       │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  Agent: Task Execution Planner                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the core skills for candidate ?Extract key skills from the provided resume using                 │
│  pdf_search_tool                                                                                                │
│  ID: bacb026a-e007-4b59-8199-069437e27cf6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyst                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│  "function": "pdf_search_tool",                                                                                 │
│  "{function\": \"search pdf documents\", parameters":                                                           │
│  "{\"query\": " Extract key skills from the provided resume using pdf_search_tool"}}                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the core skills for candidate ?Extract key skills from the provided resume using                 │
│  pdf_search_tool                                                                                                │
│  Agent: Resume Analyst                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
"function": "pdf_search_tool",
"{function\": \"search pdf documents\", parameters": 
"{\"query\": " Extract key skills from the provided resume using pdf_search_tool"}}


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f593d4fd-e677-44c0-bbb6-7ef9a12ada53                                                                       │
│  Final Output: {                                                                                                │
│  "function": "pdf_search_tool",                                                                                 │
│  "{function\": \"search pdf documents\", parameters":                                                           │
│  "{\"query\": " Extract key skills from the provided resume using pdf_search_tool"}}                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 61724642-8e4f-4457-a0de-3f0a3f60de65                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/61724642-8e4f-445 │
│ 7-a0de-3f0a3f60de65?access_code=TRACE-0721bd41ae                             │
│ 🔑 Access Code: TRACE-0721bd41ae                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


In [ ]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.ollama.types import OllamaProviderSpec

# Create a RAG tool with custom configuration

vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        #"persist_directory":"/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/vactor_store/chroma"
    }
}

embedding_model: OllamaProviderSpec = {
    "provider": "ollama",
    "config": {
        "model_name": "nomic-embed-text",
        "url": "http://localhost:11434/api/embeddings"
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model
}

rag_tool = RagTool(config=config, summarize=True)



In [ ]:
rag_tool.add("Hello")